In [ ]:
# implied_volatility.py
from datetime import date, datetime
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import yfinance as yf

In [14]:
def get_option_data(stock_symbol):
    print(f"[fetch] requesting data for {stock_symbol}")
    try:
        stock = yf.Ticker(stock_symbol)
        S = stock.fast_info["lastPrice"]
        print(f"[fetch] spot price: {S:.2f}")
    except KeyError as e:
        print(f"[fetch] ERROR - could not read spot price: {e}")
        raise
 
    try:
        exp_dates = stock.options
        print(f"[fetch] found {len(exp_dates)} expiration dates: {exp_dates[0]} ... {exp_dates[-1]}")
    except Exception as e:
        print(f"[fetch] ERROR - could not read expirations: {e}")
        raise
 
    all_calls, all_puts = [], []
    for exp in exp_dates:
        try:
            chain = stock.option_chain(exp)
            T = (datetime.strptime(exp, "%Y-%m-%d").date() - date.today()).days / 365
            all_calls.append(chain.calls.assign(T=T, expiration=exp))
            all_puts.append(chain.puts.assign(T=T, expiration=exp))
            print(f"[fetch] {exp} | T={T:.4f} | calls={len(chain.calls)} puts={len(chain.puts)}")
        except Exception as e:
            print(f"[fetch] WARNING - skipping {exp}: {e}")
            continue
 
    try:
        calls_df = pd.concat(all_calls, ignore_index=True)
        puts_df = pd.concat(all_puts, ignore_index=True)
        print(f"[fetch] total contracts - calls: {len(calls_df)}, puts: {len(puts_df)}")
    except ValueError as e:
        print(f"[fetch] ERROR - concat failed, no valid chains collected: {e}")
        raise
 
    r = 0.037
    return S, r, calls_df, puts_df


In [15]:
def bs_price(S, K, T, r, sigma, option_type):
    if T <= 0 or sigma <= 0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


In [16]:
def calc_iv(market_price, S, K, T, r, option_type):
    if T <= 0 or market_price <= 0:
        return np.nan
    intrinsic = max(0, S - K) if option_type == "call" else max(0, K - S)
    if market_price <= intrinsic:
        return np.nan
    try:
        return brentq(
            lambda sigma: bs_price(S, K, T, r, sigma, option_type) - market_price,
            1e-6, 10.0, xtol=1e-6, maxiter=500
        )
    except ValueError:
        # brentq raises ValueError when f(a) and f(b) have the same sign - no root in bracket
        return np.nan
    except RuntimeError:
        # brentq raises RuntimeError when maxiter exceeded without convergence
        return np.nan


In [22]:
def add_iv(df, S, r, option_type):
    print(f"\n[iv] calculating IV for {len(df)} {option_type} contracts...")
    df = df.copy()
 
    try:
        df["mid_price"] = (df["bid"] + df["ask"]) / 2
    except KeyError as e:
        print(f"[iv] ERROR - missing bid/ask columns: {e}")
        raise
 
    df["IV"] = df.apply(
        lambda row: calc_iv(row["mid_price"], S, row["strike"], row["T"], r, option_type),
        axis=1
    )
 
    total = len(df)
    valid = df["IV"].notna().sum()
    failed = total - valid
    print(f"[iv] {option_type} results - valid: {valid}, failed/skipped: {failed} ({100*failed/total:.1f}%)")
 
    try:
        iv_valid = df["IV"].dropna()
        print(f"[iv] {option_type} IV range - min: {iv_valid.min():.4f}, max: {iv_valid.max():.4f}, median: {iv_valid.median():.4f}")
    except Exception as e:
        print(f"[iv] WARNING - could not compute IV stats: {e}")
 
    try:
        df["IV_yf"] = df["impliedVolatility"]
        df["IV_diff"] = df["IV"] - df["IV_yf"]
        yf_valid = df["IV_yf"].notna().sum()
        print(f"[iv] yfinance IV available for {yf_valid}/{total} contracts")
    except KeyError:
        print(f"[iv] WARNING - impliedVolatility column not present in yfinance data")
 
    return df



In [23]:
def print_iv_sample(df, label, n=5):
    has_yf = "IV_yf" in df.columns
    cols = ["expiration", "strike", "T", "mid_price", "IV"] + (["IV_yf", "IV_diff"] if has_yf else [])
    try:
        sample = (
            df[cols]
            .dropna(subset=["IV"])
            .query("IV > 0.01 and IV < 5.0")
            .head(n)
        )
    except KeyError as e:
        print(f"[print] ERROR - missing expected column: {e}")
        return
 
    if sample.empty:
        print(f"\n--- {label} - no valid contracts to display ---")
        return
 
    print(f"\n--- {label} (first {n} valid contracts) ---")
    sample = sample.copy()
    sample["T"] = sample["T"].map("{:.4f}".format)
    sample["mid_price"] = sample["mid_price"].map("{:.2f}".format)
    sample["IV"] = sample["IV"].map("{:.4f}".format)
    if has_yf:
        sample["IV_yf"] = sample["IV_yf"].map(lambda x: f"{x:.4f}" if pd.notna(x) else "n/a")
        sample["IV_diff"] = sample["IV_diff"].map(lambda x: f"{x:+.4f}" if pd.notna(x) else "n/a")
    print(sample.to_string(index=False))
 
 
if __name__ == "__main__":
    S, r, calls_df, puts_df = get_option_data("AAPL")
    print(f"\n[main] spot price: {S:.2f}  |  risk-free rate: {r}")
 
    calls_df = add_iv(calls_df, S, r, "call")
    puts_df = add_iv(puts_df, S, r, "put")
 
    min_T = 7 / 365
    atm_calls = calls_df[calls_df["strike"].between(S * 0.90, S * 1.10) & (calls_df["T"] >= min_T)]
    atm_puts = puts_df[puts_df["strike"].between(S * 0.90, S * 1.10) & (puts_df["T"] >= min_T)]
    print(f"\n[main] ATM+7d filter - calls: {len(atm_calls)}, puts: {len(atm_puts)}")
 
    print_iv_sample(atm_calls, "CALLS ATM")
    print_iv_sample(atm_puts, "PUTS ATM")


[fetch] requesting data for AAPL
[fetch] spot price: 270.17
[fetch] found 23 expiration dates: 2026-04-29 ... 2028-12-15
[fetch] 2026-04-29 | T=-0.0027 | calls=38 puts=35
[fetch] 2026-05-01 | T=0.0027 | calls=59 puts=56
[fetch] 2026-05-04 | T=0.0110 | calls=28 puts=27
[fetch] 2026-05-06 | T=0.0164 | calls=20 puts=22
[fetch] 2026-05-08 | T=0.0219 | calls=52 puts=45
[fetch] 2026-05-15 | T=0.0411 | calls=88 puts=78
[fetch] 2026-05-22 | T=0.0603 | calls=39 puts=37
[fetch] 2026-05-29 | T=0.0795 | calls=37 puts=31
[fetch] 2026-06-05 | T=0.0986 | calls=22 puts=16
[fetch] 2026-06-18 | T=0.1342 | calls=81 puts=75
[fetch] 2026-07-17 | T=0.2137 | calls=52 puts=44
[fetch] 2026-08-21 | T=0.3096 | calls=59 puts=53
[fetch] 2026-09-18 | T=0.3863 | calls=79 puts=65
[fetch] 2026-10-16 | T=0.4630 | calls=49 puts=37
[fetch] 2026-11-20 | T=0.5589 | calls=62 puts=52
[fetch] 2026-12-18 | T=0.6356 | calls=85 puts=76
[fetch] 2027-01-15 | T=0.7123 | calls=72 puts=64
[fetch] 2027-03-19 | T=0.8849 | calls=58 puts